In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr

train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e5/train.csv")
y = train["PitNextLap"].astype(int).values

oof_002 = np.load(
    "/kaggle/input/datasets/mizushimatoshihiko/npy-files/oof_exp_20260501_002_cat_strict_raw_baseline.npy"
)
oof_006 = np.load(
    "/kaggle/input/datasets/mizushimatoshihiko/npy-files/oof_exp_20260501_006_cat_original_prior_min_without_normalized.npy"
)
oof_006b = np.load(
    "/kaggle/input/datasets/mizushimatoshihiko/npy-files/oof_exp_20260503_006b_cat_original_prior_no_year_prior_min.npy"
)
oof_006c = np.load(
    "/kaggle/input/datasets/mizushimatoshihiko/npy-files/oof_exp_20260503_006c_cat_original_prior_no_count_min.npy"
)

def report(name, oof):
    print(name)
    print("  AUC:", roc_auc_score(y, oof))
    print()

report("002_cat_raw", oof_002)
report("006_cat_original_prior_min", oof_006)
report("006b_cat_original_prior_no_year", oof_006b)
report("006c_cat_original_prior_no_count", oof_006c)

pairs = [
    ("002", oof_002, "006", oof_006),
    ("002", oof_002, "006b", oof_006b),
    ("002", oof_002, "006c", oof_006c),
    ("006", oof_006, "006b", oof_006b),
    ("006", oof_006, "006c", oof_006c),
    ("006b", oof_006b, "006c", oof_006c),
]

print("\nCorrelation:")
for a_name, a, b_name, b in pairs:
    pearson = np.corrcoef(a, b)[0, 1]
    spearman = spearmanr(a, b).correlation
    print(f"{a_name} vs {b_name}: pearson={pearson:.6f}, spearman={spearman:.6f}")

blend_candidates = {
    "avg_002_006": (oof_002 + oof_006) / 2,
    "avg_002_006b": (oof_002 + oof_006b) / 2,
    "avg_002_006c": (oof_002 + oof_006c) / 2,
    "avg_006_006b": (oof_006 + oof_006b) / 2,
    "avg_006b_006c": (oof_006b + oof_006c) / 2,
    "avg_002_006_006b": (oof_002 + oof_006 + oof_006b) / 3,
    "avg_002_006b_006c": (oof_002 + oof_006b + oof_006c) / 3,
    "avg_002_006_006b_006c": (oof_002 + oof_006 + oof_006b + oof_006c) / 4,
}

print("\nBlend probe:")
for name, o in blend_candidates.items():
    print(name, roc_auc_score(y, o))